# Notebook 04 — Results and Figures

**Question:** what does the detector produce for each of the three runs, and which single
figure shows it?

Generates the per-test summary dashboards and one comparison figure across all three runs,
then writes a raw detection summary.

The lead-time and downtime-cost analysis is deliberately **not** here. It needs a
sustained-alert rule and a false-alarm rate calibrated on a healthy baseline, which
`business.py` adds. The intervals this notebook prints are raw first-alert numbers and are
labelled as such.

---

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from nasa_bearing_anomaly.business import (
    PUBLISHED_HOURLY_COST,
    DowntimeCostModel,
    compute_lead_time,
    cost_sensitivity,
    format_result,
)
from nasa_bearing_anomaly.config import (
    FIGURES_DIR,
    HEADLINE_DIR,
    REPORTS_DIR,
    TEST_CONFIG,
    repo_path,
)
from nasa_bearing_anomaly.detection import run_pipeline, select_features
from nasa_bearing_anomaly.plotting import BearingPlotter

# Output directories come from config, which resolves them against the package
# location rather than the working directory. The relative literals these replaced
# ("../results/figures") were correct only when the kernel happened to start in
# notebooks/, and silently wrote elsewhere when it did not.
for d in (FIGURES_DIR, REPORTS_DIR, HEADLINE_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Ready")

## 1. Score All Three Runs

Detection is re-run here rather than read back from `results/reports/*_results.csv`. Those
scored frames are gitignored, so on a fresh clone they are absent — and a cell that reads
them when present and recomputes when not would produce different numbers depending on
what happened to be on disk.

The feature table is pinned to the **enriched** one for the same reason. It is written by
notebook 02, so run these in order; pinning it means a missing table raises instead of
silently falling back to the narrower committed table, which selects 4 feature columns
instead of 20 and fits a different model.

In [ ]:
FEATURE_SOURCE = "enriched"

results = {}
for test_id in [1, 2, 3]:
    results[test_id] = run_pipeline(test_id, feature_source=FEATURE_SOURCE)

# Computed here rather than after the figures, because the comparison figure below
# marks the alarm and must mark the sustained one. An earlier version drew the raw
# first flagged file, which is file 0 in all three runs -- so the README's hero image
# annotated "first alert: file 0 of 2156" while the text explained why that number
# means nothing.
lead_results = [
    compute_lead_time(df, test_id, feature_source=FEATURE_SOURCE) for test_id, df in results.items()
]
lead_by_test = {r.test_id: r for r in lead_results}

## 2. Summary Dashboard (Per Test)

The main portfolio image — one per test.

In [ ]:
for test_id, df in results.items():
    print(f"\nGenerating dashboard for Test {test_id}...")
    plotter = BearingPlotter(test_id=test_id, save_figures=True)
    config = TEST_CONFIG[test_id]
    feature_cols = select_features(df, bearing_prefix=config["failed_bearing"])

    # The dashboard is handed the sustained alarm rather than deriving one. It used
    # to draw df.index[is_anomaly].min(), which is file 0 on every run, so the
    # centrepiece figure annotated the one number this project's prose calls
    # meaningless. Passing it in means the figure cannot disagree with business.py.
    lead = lead_by_test[test_id]
    fig = plotter.plot_summary_dashboard(
        df,
        feature_cols=feature_cols,
        alarm_file=lead.alarm_file,
        lead_hours=lead.lead_hours,
        k=lead.k,
        m=lead.m,
    )
    plt.show()

## 3. All-Tests Comparison Figure

One clean figure showing all 3 tests side by side — for README header.

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(18, 13))
fig.patch.set_facecolor("#0d1117")
fig.suptitle(
    "NASA IMS Bearing Dataset — Predictive Maintenance via Anomaly Detection\n"
    "All Three Test Runs | Isolation Forest | Sustained-alert rule, 1% calibrated FAR",
    fontsize=13,
    fontweight="bold",
    color="#e6edf3",
    y=0.99,
)

bearing_colors = ["#58a6ff", "#3fb950", "#d29922", "#bc8cff"]

for row, (test_id, df) in enumerate(results.items()):
    config = TEST_CONFIG[test_id]
    failed = config["failed_bearing"]
    rms_col = f"{failed}_ch1_rms"
    lead = lead_by_test[test_id]

    ax_rms = axes[row, 0]
    ax_score = axes[row, 1]

    for ax in [ax_rms, ax_score]:
        ax.set_facecolor("#161b22")

    # ── Left: All bearings RMS ──
    for i, bearing in enumerate(["Bearing1", "Bearing2", "Bearing3", "Bearing4"]):
        col = f"{bearing}_ch1_rms"
        if col in df.columns:
            lw = 1.5 if bearing == failed else 0.6
            alpha = 1.0 if bearing == failed else 0.5
            lbl = bearing + (" (failed)" if bearing == failed else "")
            ax_rms.plot(
                df.index, df[col], color=bearing_colors[i], linewidth=lw, alpha=alpha, label=lbl
            )

    ax_rms.set_title(
        f"Test {test_id}: RMS — {config['failure_mode']}",
        fontsize=10,
        color="#e6edf3",
        fontweight="bold",
    )
    ax_rms.set_ylabel("RMS (g)", color="#e6edf3")
    ax_rms.legend(fontsize=7, loc="upper left")
    ax_rms.grid(True, color="#30363d", alpha=0.6)
    ax_rms.tick_params(colors="#e6edf3")

    # ── Right: Detection result, marked at the SUSTAINED alarm ──
    if rms_col in df.columns:
        ax_score.plot(df.index, df[rms_col], color="#58a6ff", linewidth=0.8, alpha=0.7, label="RMS")

        if "is_anomaly" in df.columns:
            anom_mask = df["is_anomaly"]
            ax_score.scatter(
                df.index[anom_mask],
                df[rms_col][anom_mask],
                color="#f85149",
                s=6,
                zorder=5,
                alpha=0.5,
                label="Flagged file (raw)",
            )

        # The post-shutdown tail is measured per run, not assumed to be one file.
        if lead.n_shutdown_dropped:
            ax_score.axvspan(
                df.index[len(df) - lead.n_shutdown_dropped],
                df.index[-1],
                color="#6e7681",
                alpha=0.35,
                label=f"Post-shutdown ({lead.n_shutdown_dropped})",
            )

        if lead.alarm_file is not None:
            ax_score.axvline(
                x=lead.alarm_file,
                color="#3fb950",
                linestyle="--",
                linewidth=1.8,
                label=f"Sustained alarm ({lead.k}-of-{lead.m})",
            )
            ax_score.text(
                lead.alarm_file - max(2, len(df) * 0.01),
                ax_score.get_ylim()[1] * 0.70 if ax_score.get_ylim()[1] > 0 else 0.5,
                f"{lead.lead_hours:.0f} h warning\nfile {lead.alarm_file} of {len(df)}",
                color="#3fb950",
                fontsize=8,
                fontweight="bold",
                ha="right",
            )

        ax_score.axvline(
            x=lead.anchor_file, color="#f85149", linestyle="-", linewidth=1.5, label="Failure"
        )

    ax_score.set_title(
        f"Test {test_id}: {lead.lead_hours:.0f} h warning, 0 false alarms in "
        f"{lead.heldout_files} healthy files",
        fontsize=10,
        color="#e6edf3",
        fontweight="bold",
    )
    ax_score.set_ylabel(f"{failed} RMS (g)", color="#e6edf3")
    ax_score.legend(fontsize=7, loc="upper left")
    ax_score.grid(True, color="#30363d", alpha=0.6)
    ax_score.tick_params(colors="#e6edf3")

for ax in axes[-1]:
    ax.set_xlabel("File Index (Time →)", color="#e6edf3")

plt.tight_layout()
out_path = FIGURES_DIR / "all_tests_comparison.png"
plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor="#0d1117")
print(f"Saved: {repo_path(out_path)}")
plt.show()

## 4. Lead Time and Its False-Alarm Cost

`business.py` owns this metric; the cell below only renders it. That split is deliberate —
an earlier version of this notebook computed lead time inline, and an inline copy of a
metric drifts from the module until the two report different numbers with equal confidence.

What the module does, and why each step is needed, is in its docstring. In short: the
`is_anomaly` column cannot be used as an alarm, because `contamination` fixes the fraction
of *healthy* training files labelled anomalous, so a metric triggering on the first flag
just measures the length of the run. The threshold is instead calibrated on healthy data to
a chosen false-alarm rate, a k-of-m sustained rule is chosen against healthy data alone, and
the post-shutdown tail is measured rather than assumed to be one file.

In [ ]:
print("LEAD TIME AND FALSE-ALARM COST")
print("=" * 78)
for r in lead_results:
    print(format_result(r))
    print()

summary = pd.DataFrame([r.to_row() for r in lead_results])
summary.to_csv(REPORTS_DIR / "business_summary.csv", index=False)
print("Saved: results/reports/business_summary.csv")

### What the warning is worth

Lead time is **not** multiplied by an hourly rate. That prices hours during which the
machine was still running normally, and it is the mistake the earlier inline version made.

Warning converts an *unplanned* stoppage into a *planned* one, so the saving is the
difference between the two stoppage lengths — and it is realised only if the warning is long
enough to order the part and book a window. Past that point a longer warning does not save
more money, which is why the figures below are flat across all three tests.

Every number in the model is an assumption, not a measurement, so the rate is varied rather
than asserted. The published whole-facility figures printed underneath are there for scale
only: a single bearing is not a whole plant, and the gap between the two is the point.

In [ ]:
model = DowntimeCostModel()
print(f"Model assumptions: {model.assumptions()}\n")

cost_tables = []
for r in lead_results:
    table = cost_sensitivity(r.lead_hours, model=model)
    table.insert(0, "test", r.test_id)
    cost_tables.append(table)

cost_df = pd.concat(cost_tables, ignore_index=True)
print(cost_df.to_string(index=False))
cost_df.to_csv(REPORTS_DIR / "business_cost_sensitivity.csv", index=False)
print("\nSaved: results/reports/business_cost_sensitivity.csv\n")

print("For scale -- published whole-facility rates, NOT the rate used above:")
for label, usd in PUBLISHED_HOURLY_COST.items():
    print(f"  USD {usd:>9,.0f}/h  {label}")

## 5. Write the README Hero Figure

`figures/headline/` holds only the figures the README embeds, matching the rest of the
portfolio. Everything else stays under `results/figures/`.

In [ ]:
import shutil

src = FIGURES_DIR / "all_tests_comparison.png"
dst = HEADLINE_DIR / "01_all_tests_comparison.png"
if src.exists():
    shutil.copy(src, dst)
    print(f"Saved: {repo_path(dst)}")
    print("   Embedded in README.md as: ![Results](figures/headline/01_all_tests_comparison.png)")